# Day 13. Scaled Dot-Product Attention

**오늘의 목표**
- `softmax(QKᵀ / √dk) V`를 단계별로 설명할 수 있다
- **왜 `√dk`로 나누는지** 실험으로 직접 확인한다 (분산, Softmax 쏠림, Gradient 소실)
- Attention 함수를 처음부터 구현하고, PyTorch 공식 함수와 결과가 같은지 확인한다
- `Head(nn.Module)`로 감싸서 mini GPT에서 쓸 형태로 만든다 (커리큘럼 6-4 미리보기)

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

## 1. 전체 공식 다시 보기

```text
Attention(Q, K, V) = softmax(QKᵀ / √dk) V
```

Day 12까지는 `softmax(QKᵀ) V`, 즉 **Scale 없이** 계산했습니다. 오늘은 빠져 있던 `/ √dk`를 채웁니다.

```text
Q × Kᵀ            (T, dk) @ (dk, T) → (T, T)   Token 쌍마다 관련도 점수
 ↓
Scale  (÷ √dk)    (T, T)                       점수 크기를 적당하게 조절  ← 오늘의 주인공
 ↓
Softmax           (T, T)                       행마다 합이 1인 비율로
 ↓
Attention Weight  (T, T)                       "누구를 몇 % 참고할까"
 ↓
× V               (T, T) @ (T, dk) → (T, dk)   참고 비율대로 정보를 섞기
 ↓
Output            (T, dk)
```

`dk`는 Query/Key 벡터 하나의 길이(= `head_size`)입니다.

## 2. 왜 나누나? ① 점수의 분산이 dk만큼 커진다

Q와 K의 각 원소가 평균 0, 분산 1인 랜덤 값이라고 해봅시다. (학습 초기의 상태와 비슷해요)

Dot Product는 `q₁k₁ + q₂k₂ + ... + q_dk·k_dk`, 즉 **dk개를 더한 값**입니다.
분산이 1인 값을 dk개 더하면 분산은 대략 **dk**가 됩니다.

직접 재봅시다.

In [ ]:
dks = [4, 16, 64, 256]
variances = []

for dk in dks:
    q = torch.randn(10000, dk)
    k = torch.randn(10000, dk)
    scores = (q * k).sum(dim=-1)          # 10000쌍의 Dot Product
    variances.append(scores.var().item())
    print(f"dk = {dk:4d}  →  점수의 분산 ≈ {scores.var().item():7.2f}   표준편차 ≈ {scores.std().item():5.2f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(dks, variances, "o-", linewidth=2.5, markersize=8, label="측정한 분산")
plt.plot(dks, dks, "--", color="gray", label="y = dk")
plt.xlabel("dk (head_size)")
plt.ylabel("$QK^T$ 점수의 분산")
plt.title("dk가 커질수록 점수의 분산도 커진다", fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

측정값이 회색 점선 `y = dk` 위에 거의 그대로 올라갑니다.

`dk = 256`이면 표준편차가 약 16이라서 점수가 **-30 ~ +30** 정도로 크게 흩어집니다.
실제 GPT-2의 head_size가 64이니, 이건 실제로 일어나는 문제예요.

## 3. 왜 나누나? ② 점수가 크면 Softmax가 한쪽으로 쏠린다

Day 5의 **Temperature**를 떠올려보세요. logits 차이가 크면(Temperature가 낮으면) Softmax가 1등에 몰빵했죠.

점수의 분산이 크다 = 점수 차이가 크다 = **Softmax가 거의 one-hot**이 됩니다.
`dk = 256`에서 Query 하나가 Token 8개를 보는 상황을 만들어 봅시다.

In [ ]:
torch.manual_seed(3)
dk, T = 256, 8
q = torch.randn(dk)
k = torch.randn(T, dk)

raw    = k @ q                     # (T,) 스케일 없는 점수
scaled = raw / math.sqrt(dk)       # (T,) √dk로 나눈 점수

w_raw    = F.softmax(raw, dim=-1)
w_scaled = F.softmax(scaled, dim=-1)

print("스케일 없는 점수 :", r(raw, 1))
print("√dk로 나눈 점수  :", r(scaled, 2))
print()
print("Weight (스케일 X):", r(w_raw, 3))
print("Weight (스케일 O):", r(w_scaled, 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
labels = [f"T{i}" for i in range(T)]
for ax, w, title in [
    (axes[0], w_raw,    "스케일 없음: $softmax(QK^T)$"),
    (axes[1], w_scaled, r"스케일 있음: $softmax(QK^T / \sqrt{d_k})$"),
]:
    ax.bar(labels, w, color="#4C72B0")
    for i, v in enumerate(w):
        ax.text(i, v.item() + 0.02, f"{v.item()*100:.0f}%", ha="center", fontsize=9)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(0, 1.1)
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Attention Weight")
plt.suptitle("dk = 256일 때 Query 하나의 Attention Weight", fontsize=13)
plt.tight_layout()
plt.show()

스케일이 없으면 Weight가 한두 Token에 거의 전부 몰립니다. 나머지 Token의 정보는 사실상 무시돼요.
Attention의 장점은 "여러 Token을 **적당히 섞어서** 참고하는 것"인데, 그 장점이 사라집니다.

> 💡 `/ √dk`는 **Temperature = √dk**로 Softmax를 하는 것과 똑같습니다. (Day 5: `softmax(logits / T)`)

## 4. 왜 나누나? ③ 쏠린 Softmax는 학습이 안 된다 (Gradient 소실)

Softmax가 거의 one-hot이면, 점수를 조금 바꿔도 결과가 거의 안 바뀝니다. (이미 100%라 더 올라갈 데가 없음)
결과가 안 바뀐다 = **Gradient가 거의 0** = Day 7에서 배운 Weight 업데이트가 거의 일어나지 않음.

**Softmax에 들어가는 점수**에 대한 Gradient 크기를 200번 측정해서 평균을 내봅시다.
(Softmax가 Gradient를 얼마나 잘 통과시키는지만 보려고, 크기가 고정된 아무 Loss를 하나 씁니다)

In [ ]:
def measure(dk, scale, trials=200, T=8):
    grad_total, peak_total = 0.0, 0.0
    for _ in range(trials):
        q = torch.randn(T, dk)
        k = torch.randn(T, dk)
        scores = q @ k.T
        if scale:
            scores = scores / math.sqrt(dk)
        scores = scores.detach().requires_grad_(True)   # Softmax 입력에 대한 Gradient를 보기 위해

        weights = F.softmax(scores, dim=-1)
        loss = (weights * torch.randn(T, T)).sum()       # 아무 Loss나 하나
        loss.backward()

        grad_total += scores.grad.norm().item()
        peak_total += weights.max(dim=-1).values.mean().item()   # 1등 Token의 평균 Weight
    return grad_total / trials, peak_total / trials

torch.manual_seed(0)
print("             |  1등 Token Weight 평균  |  Softmax 입력의 Gradient 크기")
print("             |  스케일 X   스케일 O    |  스케일 X   스케일 O")
for dk in dks:
    g_raw, p_raw = measure(dk, scale=False)
    g_scl, p_scl = measure(dk, scale=True)
    print(f"dk = {dk:4d}   |  {p_raw*100:6.1f}%   {p_scl*100:6.1f}%     |  {g_raw:7.3f}   {g_scl:7.3f}")

**스케일이 없으면** dk가 커질수록 1등 Token에 Weight가 몰리고(one-hot에 가까워지고), Gradient가 점점 작아져서 학습 신호가 약해집니다.
**스케일이 있으면** dk와 상관없이 Weight 분포와 Gradient 크기가 비슷하게 유지됩니다.

> 💡 정리: `√dk`로 나누는 이유는 **점수의 분산을 1 근처로 맞춰서**, Softmax가 쏠리지 않고 Gradient가 잘 흐르게 하기 위해서입니다.

## 5. 나누면 정말 분산이 1이 되나?

분산의 성질: 어떤 값을 `c`로 나누면 분산은 `c²`로 나눠집니다.
`√dk`로 나누면 분산은 `dk / (√dk)² = dk / dk = 1`이 됩니다.

In [ ]:
for dk in dks:
    q = torch.randn(10000, dk)
    k = torch.randn(10000, dk)
    scores = (q * k).sum(dim=-1)
    print(f"dk = {dk:4d}   스케일 전 분산: {scores.var().item():7.2f}   "
          f"√dk로 나눈 후 분산: {(scores / math.sqrt(dk)).var().item():5.2f}")

dk가 얼마든 **분산 ≈ 1**로 맞춰집니다.

> 💡 이렇게 "값의 크기(분산)를 일정하게 유지해서 학습을 안정시키는" 아이디어는 Day 16 **LayerNorm**에서 다시 나옵니다.

## 6. Attention 함수 직접 구현하기

이제 공식 전체를 함수 하나로 만듭니다. 각 단계의 shape를 출력하게 해서 따라가 봐요.
`mask`는 Day 14에서 자세히 다루고, 오늘은 "있으면 `-inf`로 가린다"는 것만 넣어둡니다. (Day 5 8-1에서 본 트릭)

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None, verbose=False):
    dk = q.shape[-1]

    scores = q @ k.transpose(-2, -1)                   # 1) Q × Kᵀ
    scores = scores / math.sqrt(dk)                    # 2) Scale
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))   # (Day 14) 가리기
    weights = F.softmax(scores, dim=-1)                # 3) Softmax → Attention Weight
    out = weights @ v                                  # 4) × V

    if verbose:
        print("q       :", tuple(q.shape))
        print("k       :", tuple(k.shape))
        print("v       :", tuple(v.shape))
        print("scores  :", tuple(scores.shape), " ← Q × Kᵀ / √dk")
        print("weights :", tuple(weights.shape), " ← Softmax, 행마다 합 1")
        print("out     :", tuple(out.shape), " ← weights × V")
    return out, weights

### 6-1. 예시 문장으로 돌려보기

"나는 오늘 학교에 간다" (T = 4), head_size(dk) = 8

In [ ]:
torch.manual_seed(0)
tokens = ["나는", "오늘", "학교에", "간다"]
T, dk = len(tokens), 8

q = torch.randn(T, dk)
k = torch.randn(T, dk)
v = torch.randn(T, dk)

out, weights = scaled_dot_product_attention(q, k, v, verbose=True)

print()
print("Attention Weight (행 = 지금 Token, 열 = 참고하는 Token)")
print("        " + "".join(f"{t:>8s}" for t in tokens))
for t, row in zip(tokens, weights):
    print(f"{t:>6s}  " + "".join(f"{x.item()*100:7.1f}%" for x in row))
print()
print("행마다 합:", r(weights.sum(dim=-1), 4))

`out`의 각 행은 **해당 Token이 다른 Token들의 V를 Weight 비율대로 섞은 새 벡터**입니다.
첫 번째 행이 정말 그런지 직접 확인해봅시다.

In [ ]:
manual_row0 = sum(weights[0, j] * v[j] for j in range(T))
print("out[0]          :", r(out[0], 3))
print("직접 섞은 결과  :", r(manual_row0, 3))
print("같은가?", torch.allclose(out[0], manual_row0, atol=1e-6))

## 7. PyTorch 공식 함수와 비교하기

PyTorch 2.x에는 `F.scaled_dot_product_attention`이 있습니다. 내부적으로 훨씬 빠르게 계산하지만, **수학적으로는 똑같은 일**을 합니다.
batch 축이 있는 `(B, T, dk)`로 비교해봅시다.

In [ ]:
torch.manual_seed(0)
B, T, dk = 2, 4, 8
q = torch.randn(B, T, dk)
k = torch.randn(B, T, dk)
v = torch.randn(B, T, dk)

mine, _ = scaled_dot_product_attention(q, k, v, verbose=True)
official = F.scaled_dot_product_attention(q, k, v)

print()
print("내 구현     :", tuple(mine.shape))
print("공식 함수   :", tuple(official.shape))
print("결과가 같은가?", torch.allclose(mine, official, atol=1e-6))

mask까지 넣어도 같은지 확인해봅시다. (공식 함수는 `is_causal=True`로 미래 가리기를 해줘요. 자세한 건 Day 14)

In [ ]:
causal = torch.tril(torch.ones(T, T))
mine_masked, w_masked = scaled_dot_product_attention(q, k, v, mask=causal)
official_masked = F.scaled_dot_product_attention(q, k, v, is_causal=True)

print("mask 적용 결과가 같은가?", torch.allclose(mine_masked, official_masked, atol=1e-6))
print()
print("첫 번째 문장의 Weight (위쪽 삼각형이 0):")
print(w_masked[0].round(decimals=3))

## 8. `Head` 모듈로 감싸기 (커리큘럼 6-4 미리보기)

실제 모델에서는 Q, K, V를 입력 `x`로부터 **Linear Layer로 만들어서** (Day 3, Day 12) Attention에 넣습니다.
이걸 `nn.Module` 하나로 묶은 것이 **Attention Head** 하나예요.

In [ ]:
class Head(nn.Module):
    # Self-Attention Head 하나

    def __init__(self, n_embd, head_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

    def forward(self, x, mask=None):
        q = self.query(x)     # (B, T, head_size)
        k = self.key(x)       # (B, T, head_size)
        v = self.value(x)     # (B, T, head_size)
        out, _ = scaled_dot_product_attention(q, k, v, mask=mask)
        return out            # (B, T, head_size)

B, T, n_embd, head_size = 2, 4, 32, 16
x = torch.randn(B, T, n_embd)          # Embedding 결과라고 생각하세요

head = Head(n_embd, head_size)
out = head(x)

print("입력 x :", tuple(x.shape), "  (B, T, n_embd)")
print("출력   :", tuple(out.shape), "  (B, T, head_size)")
print("파라미터 개수:", sum(p.numel() for p in head.parameters()), f"= 3 × ({n_embd} × {head_size})")

## 9. LLM과의 연결

GPT 안에서 오늘 만든 함수가 쓰이는 위치:

```text
Token → Embedding (Day 9) + Position (Day 10)
  ↓
┌ Transformer Block ──────────────────────────┐
│  Multi-Head Attention (Day 14)              │
│    = Head × 여러 개                          │
│    = 각 Head 안에서 softmax(QKᵀ/√dk)V  ← 오늘 │
│  FFN (Day 4, Day 17)                        │
└─────────────────────────────────────────────┘
  ↓ × N
lm_head → Softmax → 다음 Token
```

- **Scale(`/ √dk`)** 은 학습이 안정적으로 되게 해주는 장치입니다. 없어도 계산은 되지만, 큰 모델에서는 학습이 잘 안 돼요.
- 내일(Day 14)은 오늘 만든 `Head`에 **Causal Mask**를 정식으로 넣고, 여러 개를 병렬로 돌리는 **Multi-Head Attention**을 만듭니다.

## 10. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** `dk = 100`일 때, Q와 K의 원소가 평균 0·분산 1인 랜덤 값이면 `q · k` 점수의 분산은 대략 얼마일까요?
실행하기 **전에** 가장 가까운 것을 고르세요: `"약 1"`, `"약 10"`, `"약 100"`

In [ ]:
my_answer = "?"   # TODO

q = torch.randn(10000, 100)
k = torch.randn(10000, 100)
measured = (q * k).sum(dim=-1).var().item()

def closest(x):
    return min(["약 1", "약 10", "약 100"], key=lambda s: abs({"약 1": 1, "약 10": 10, "약 100": 100}[s] - x))

assert my_answer == closest(measured), "2절의 실험 결과를 다시 떠올려보세요"
print(f"Q1 정답! 측정한 분산 = {measured:.1f}")

**Q2.** `scores`를 **분산이 1 근처**가 되도록 스케일하세요. `dk`는 `q.shape[-1]`로 구할 수 있어요.

In [ ]:
q = torch.randn(5000, 64)
k = torch.randn(5000, 64)
scores = (q * k).sum(dim=-1)

scaled_scores = None  # TODO

assert scaled_scores is not None, "scaled_scores를 채워주세요"
assert 0.85 < scaled_scores.var().item() < 1.15, "분산이 1 근처가 아니에요"
ratio = scaled_scores / scores
assert torch.allclose(ratio, ratio[0].expand_as(ratio), atol=1e-4), "모든 점수를 같은 수로 나눠야 해요"
print(f"Q2 정답! 분산 = {scaled_scores.var().item():.3f}")

**Q3.** 보지 않고 Attention 함수를 처음부터 다시 구현하세요. (mask 없이)
- 입력: `q, k, v` 모두 `(B, T, dk)`
- 출력: `(B, T, dk)`
- 힌트: Q × Kᵀ → ÷ √dk → Softmax → × V. batch가 있으니 `.T` 대신 `transpose(-2, -1)`

In [ ]:
def my_attention(q, k, v):
    return None  # TODO

q = torch.randn(3, 5, 16)
k = torch.randn(3, 5, 16)
v = torch.randn(3, 5, 16)

result = my_attention(q, k, v)
assert result is not None, "my_attention을 채워주세요"
assert tuple(result.shape) == (3, 5, 16), "출력 shape가 달라요"
assert torch.allclose(result, F.scaled_dot_product_attention(q, k, v), atol=1e-5), "공식 함수와 결과가 달라요"
print("Q3 정답!")

**Q4.** `dk = 512`인 모델에서 **`/ √dk`를 빼먹으면** 학습 초기에 어떤 일이 생길까요?

- `"A"`: Attention Weight가 모든 Token에 고르게 퍼지고, Gradient가 커진다
- `"B"`: Attention Weight가 Token 하나에 몰리고, Gradient가 매우 작아진다
- `"C"`: 결과 shape가 `(T, T)`에서 `(T, dk)`로 바뀐다

In [ ]:
my_answer = "?"   # TODO: "A", "B", "C" 중 하나

torch.manual_seed(0)
g_raw, p_raw = measure(512, scale=False)    # 4절에서 만든 함수
g_scl, p_scl = measure(512, scale=True)

peaked = p_raw > 0.9
tiny_grad = g_raw < 0.5 * g_scl
shape_changed = tuple((torch.randn(8, 512) @ torch.randn(8, 512).T).shape) != (8, 8)
truth = "C" if shape_changed else ("B" if peaked and tiny_grad else "A")

assert my_answer == truth, "3절과 4절의 실험을 다시 떠올려보세요"
print("Q4 정답!")

## 11. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. `softmax(QKᵀ / √dk) V`의 각 단계(QKᵀ, ÷√dk, softmax, ×V)는 각각 무엇을 하는가?
2. dk가 커지면 QKᵀ 점수의 분산은 어떻게 되는가? 왜 그런가?
3. 점수의 분산이 크면 Softmax 결과는 어떻게 되는가? Day 5의 Temperature와 어떤 관계인가?
4. Softmax가 한쪽으로 쏠리면 학습에 어떤 문제가 생기는가?
5. 왜 하필 `dk`가 아니라 `√dk`로 나누는가?
6. Attention의 출력 `(T, dk)`의 각 행은 무엇을 의미하는가?

**내 답:**

1.
2.
3.
4.
5.
6.